In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
dbutils.widgets.text('externalfiles_zone', 'None', '07-Externalfiles Zone')
externalfiles_zone = dbutils.widgets.get('externalfiles_zone')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_ext_dod_dpd_fcy_extfile(
    spark, curated_zone, catalog_landed, reference_zone, externalfiles_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, externalfiles_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #      "ExtDOD_DPD_FCYId  string,"
        #      "SourceSystem   string,"
        #      "PortfolioKey   string,"
        #      "PortfolioName  string,"
        #      "IngestionTimestamp timestamp,"
        #      "BatchId    string,"
        #      "ExtDOD_DPD_FCYKey string,"
        #      "ExposureKey    string,"
        #      "ExposureSourceSystem   string,"
        #      "DODDPDStartDate    string,"
        #      "DODDPDEndDate  string,"
        #      "BusinessDate  date"
        # )
        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.ExtDOD_DPD_FCY"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "extfile",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== dod_dpd_fcy ==================
        dod_dpd_fcy_extra_filters = None

        if dod_dpd_fcy_extra_filters is not None:
            dod_dpd_fcy_extra_filters.append(date_filter)
            dod_dpd_fcy_final_filters = " AND ".join(dod_dpd_fcy_extra_filters)
        else:
            dod_dpd_fcy_final_filters = date_filter

            dod_dpd_fcy_selected_fields = [
                "AC_CODE",
                "AC_SRC_STM_CODE",
                "DOD_DPD_CNTER_STRT_DT",
                "DOD_DPD_CNTER_END_DT",
                "businessdate",
            ]

            dod_dpd_fcy = load_landed_delta_table(
                spark,
                externalfiles_zone,  # type: ignore 
                "extdod_dpd_fcy",
                dod_dpd_fcy_final_filters,
                dod_dpd_fcy_selected_fields,
            )

        # ================== dod_dpd_fcy_extfile_df ==================
        dod_dpd_fcy_extfile_primary_key_fields = [
            dod_dpd_fcy.AC_CODE,
            dod_dpd_fcy.DOD_DPD_CNTER_STRT_DT,
            dod_dpd_fcy.AC_SRC_STM_CODE,
            lit("extfile"),
        ]
        dod_dpd_fcykey_extfile_primary_key_fields = [
            dod_dpd_fcy.AC_CODE,
            dod_dpd_fcy.DOD_DPD_CNTER_STRT_DT,
            dod_dpd_fcy.AC_SRC_STM_CODE,
        ]

        dod_dpd_fcy_extfile_df = (
            dod_dpd_fcy.withColumn(
                "ExtDOD_DPD_FCYId",
                sha2(concat_ws(";", *dod_dpd_fcy_extfile_primary_key_fields), 256),
            )
            .withColumn("SourceSystem", lit("extfile").cast("string"))
            .withColumn("BatchId", lit(None).cast("string"))
            .withColumn(
                "PortfolioKey",
                (lit("-9999")),
            )
            .withColumn(
                "PortfolioName",
                (lit("Core Retail")),
            )
            .withColumn(
                "ExtDOD_DPD_FCYKey",
                concat_ws(";", *dod_dpd_fcykey_extfile_primary_key_fields),
            )
            .withColumn("dod_dpd_fcyStatus", lit(None).cast("string"))
        )

        dod_dpd_fcy_extfile_df = dod_dpd_fcy_extfile_df.select(
            dod_dpd_fcy_extfile_df.ExtDOD_DPD_FCYId.cast("string").alias(
                "ExtDOD_DPD_FCYId"
            ),
            dod_dpd_fcy_extfile_df.SourceSystem,
            dod_dpd_fcy_extfile_df.BatchId,
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            dod_dpd_fcy_extfile_df.PortfolioKey.cast("string"),
            dod_dpd_fcy_extfile_df.PortfolioName.cast("string"),
            dod_dpd_fcy_extfile_df.ExtDOD_DPD_FCYKey.cast("string").alias(
                "ExtDOD_DPD_FCYKey"
            ),
            dod_dpd_fcy_extfile_df.AC_CODE.cast("string").alias("ExposureKey"),
            dod_dpd_fcy_extfile_df.AC_SRC_STM_CODE.cast("string").alias(
                "ExposureSourceSystem"
            ),
            dod_dpd_fcy_extfile_df.DOD_DPD_CNTER_STRT_DT.cast("string").alias(
                "DODDPDStartDate"
            ),
            dod_dpd_fcy_extfile_df.DOD_DPD_CNTER_END_DT.cast("string").alias(
                "DODDPDEndDate"
            ),
            dod_dpd_fcy_extfile_df.businessdate.alias("BusinessDate"),
        )

        # ================== Append ==================

        dod_dpd_fcy_extfile_df.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)
    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                externalfiles_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_ext_dod_dpd_fcy_extfile(spark, curated_zone, catalog_landed, reference_zone, externalfiles_zone, run_date, schema, env_prefix)